# Trabalho 2 – Algoritmos Genéticos: Caixeiro Viajante

**Integrantes:** Fabian Formento, Lucas Visconti, Cauã F. Bertolini, Luiz C. Martendal, Eduarda D. Mazureck

Objetivo: encontrar, com um algoritmo genético, a rota mais curta que visita 20 cidades exatamente uma vez e volta à cidade inicial.

In [1]:
import numpy as np  # cálculos vetoriais e números aleatórios

## 1. Função de aptidão (fitness)

### 1.1 Carregando as cidades
O arquivo `cidades.mat` é texto puro, por isso usamos `np.loadtxt` (o `scipy.io.loadmat` não consegue lê-lo).
A linha 0 contém as coordenadas `x` e a linha 1 as coordenadas `y`.

In [2]:
# Carrega as coordenadas: matriz 2 x 20
x, y = np.loadtxt('cidades.mat')

Ncidades = len(x)  # número de cidades (20)
print('Número de cidades:', Ncidades)

Número de cidades: 20


### 1.2 Matriz de distâncias
Calculamos uma única vez a distância euclidiana entre todos os pares de cidades:

$$d(c_i, c_j) = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}$$

A matriz é simétrica, pois $d(c_i, c_j) = d(c_j, c_i)$.

In [3]:
def matriz_distancias(x, y):
    # Diferenças entre todas as coordenadas (broadcasting gera matrizes n x n)
    dx = x[:, None] - x[None, :]
    dy = y[:, None] - y[None, :]
    # Distância euclidiana entre cada par de cidades
    return np.sqrt(dx**2 + dy**2)

dcidade = matriz_distancias(x, y)  # matriz 20 x 20

### 1.3 Função de aptidão
Cada cromossomo é uma permutação das cidades. Internamente as cidades são numeradas de **0 a 19** (índices do Python);
na exibição dos resultados somamos 1 para ficar de 1 a 20, como no enunciado.

A aptidão é a distância total do percurso, incluindo a volta à cidade inicial ($c_{n+1} = c_1$):

$$D = \sum_{k=1}^{n} d(c_k, c_{k+1})$$

Quanto **menor** o valor de $D$, **mais apto** é o cromossomo.

In [4]:
def aptidao(pop, dcidade=dcidade):
    # Garante formato 2D: (Npop, Ncidades), mesmo se vier um único cromossomo
    pop = np.atleast_2d(pop)
    # Acrescenta a primeira cidade no final de cada rota (volta ao início),
    # equivalente ao tour = [pop pop(:,1)] do Matlab
    tour = np.hstack([pop, pop[:, :1]])
    # Distância de cada trecho c_k -> c_k+1 de cada cromossomo
    trechos = dcidade[tour[:, :-1], tour[:, 1:]]
    # Soma os trechos: distância total de cada cromossomo
    return trechos.sum(axis=1)

### 1.4 Testes da função de aptidão

In [5]:
# Teste 1: quatro cidades nos cantos de um quadrado 1 x 1
xq = np.array([0, 1, 1, 0])
yq = np.array([0, 0, 1, 1])
dq = matriz_distancias(xq, yq)
assert np.isclose(aptidao([0, 1, 2, 3], dq)[0], 4.0)                # contorno do quadrado = 4
assert np.isclose(aptidao([0, 2, 1, 3], dq)[0], 2 + 2*np.sqrt(2))  # rota cruzando as diagonais

# Teste 2: comparação com a versão em laços (igual ao código Matlab da professora)
def aptidao_lacos(pop):
    Npop, Nc = pop.shape
    tour = np.hstack([pop, pop[:, :1]])
    dist = np.zeros(Npop)
    for i in range(Npop):
        for j in range(Nc):
            dist[i] += dcidade[tour[i, j], tour[i, j + 1]]
    return dist

rng = np.random.default_rng(0)
pop_teste = np.array([rng.permutation(Ncidades) for _ in range(20)])  # 20 rotas aleatórias
assert np.allclose(aptidao(pop_teste), aptidao_lacos(pop_teste))

# Teste 3: rotacionar ou inverter a rota não muda a distância
rota = pop_teste[0]
assert np.isclose(aptidao(rota)[0], aptidao(np.roll(rota, 5))[0])
assert np.isclose(aptidao(rota)[0], aptidao(rota[::-1])[0])

# Teste 4: a matriz de distâncias é simétrica e tem diagonal zero
assert np.allclose(dcidade, dcidade.T)
assert np.allclose(np.diag(dcidade), 0)

print('Todos os testes passaram!')
print('Aptidão das 5 primeiras rotas aleatórias:', np.round(aptidao(pop_teste)[:5], 4))

Todos os testes passaram!
Aptidão das 5 primeiras rotas aleatórias: [ 9.7768 12.2454  8.8427 10.218  10.5802]


## 2. População inicial

### 2.1 Parâmetros do algoritmo
Valores definidos no enunciado: população de 20 cromossomos e 10.000 iterações.
A `SEMENTE` fixa os números aleatórios para que os resultados possam ser reproduzidos; use `None` para obter resultados diferentes a cada execução.

In [6]:
Npop = 20           # tamanho da população (número de cromossomos)
Niteracoes = 10000  # número de gerações (iterações)
SEMENTE = 42        # semente aleatória (None = resultado diferente a cada execução)

rng = np.random.default_rng(SEMENTE)  # gerador de números aleatórios usado no algoritmo

### 2.2 Gerando a população
Cada cromossomo é uma permutação aleatória das cidades `0..19`. Assim, toda rota já nasce válida:
cada cidade aparece exatamente uma vez.

In [7]:
def populacao_inicial(Npop, Ncidades):
    # Cria Npop permutações aleatórias das cidades: matriz (Npop, Ncidades)
    return np.array([rng.permutation(Ncidades) for _ in range(Npop)])

pop = populacao_inicial(Npop, Ncidades)  # população que vai evoluir
pop_inicial = pop.copy()                 # cópia guardada para exibir no final

# Mostra a população inicial (cidades de 1 a 20) e a distância de cada rota
custos = aptidao(pop)
for i, (rota, custo) in enumerate(zip(pop_inicial + 1, custos), start=1):
    print(f'{i:2d}: {rota}  D = {custo:.4f}')

 1: [16 10 15  8 13 11  7 20  4  1 17  6 12 19  3  5 18  2 14  9]  D = 10.1380
 2: [19  9 10  8  5  2  7 12  6 16  1 13 20  3 15 17 11  4 14 18]  D = 9.6963
 3: [13  8  7 14  9 18  6  2 11 10 20 15 16 19  4 12  3  1  5 17]  D = 10.3036
 4: [14  4 19  3  9 16  1  7 20 12 17 15  6 18  5  2 11  8 10 13]  D = 8.4142
 5: [16 17  3 11  9  1  7 19  5 10 13  6 15 20 14 12  2 18  4  8]  D = 11.3014
 6: [ 3 10  5 19 17 12  6  4  1 18  2  7 16  9 13 20 15 11 14  8]  D = 10.9698
 7: [ 9  4 19 17 16 13  5  7  6  1 10 15 12 20 11  3 18  8  2 14]  D = 8.0649
 8: [ 9 20  8 13  7  6 12 18  4 14 19 15 16  2  1  5 17 10  3 11]  D = 9.1427
 9: [ 7 19 17 16  8  5 11  2 10 15 18  4 20  9 12  6 14  1  3 13]  D = 10.9852
10: [ 7 15  8  6  4  5  2 17  9  1 18 19 16 14 20 12 13  3 10 11]  D = 9.8104
11: [16 17 12 10 19  4  8  5 11  1 13 20  9  2  7  6 18  3 15 14]  D = 10.4451
12: [ 5 14  7 10  4  2 11  6  1  3 18 12 19  9 15 13 20 16 17  8]  D = 8.9057
13: [ 1  9  8  3  5 14 12  4 17 11  2 19 10 18 20 16  7  6

### 2.3 Testes da população inicial

In [8]:
# Formato correto: 20 cromossomos com 20 cidades cada
assert pop.shape == (Npop, Ncidades)

# Cada cromossomo é uma permutação válida: contém todas as cidades, sem repetição
for rota in pop:
    assert np.array_equal(np.sort(rota), np.arange(Ncidades))

# A cópia é independente: alterar pop não altera pop_inicial
assert not np.shares_memory(pop, pop_inicial)

print('Todos os testes passaram!')
print(f'Melhor rota inicial: D = {custos.min():.4f} | Pior: D = {custos.max():.4f}')

Todos os testes passaram!
Melhor rota inicial: D = 8.0649 | Pior: D = 11.3014


## 3. Seleção e roleta

### 3.1 Ordenação e sobrevivência
A cada geração a população é ordenada pela aptidão em ordem **crescente** (menor distância = mais apto).
Apenas a metade melhor (`Nmanter = 10` cromossomos) sobrevive e serve de pais para a próxima geração.

In [9]:
Nmanter = Npop // 2  # número de cromossomos mantidos a cada geração (10)

def ordenar(pop):
    # Calcula a distância de cada cromossomo e ordena do menor para o maior custo
    custos = aptidao(pop)
    ordem = np.argsort(custos)
    return pop[ordem], custos[ordem]

### 3.2 Probabilidades da roleta
A probabilidade de cada sobrevivente ser escolhido como pai é inversamente proporcional à sua posição no ranking:

$$p_i = \frac{N_{manter} - i}{\sum_{k=1}^{N_{manter}} k}, \quad i = 0, \dots, N_{manter}-1$$

Com 10 sobreviventes, o melhor recebe $10/55$ e o pior recebe $1/55$.

In [10]:
# Pesos 10, 9, ..., 1 divididos pela soma (55)
pesos = np.arange(Nmanter, 0, -1)
prob = pesos / pesos.sum()

print('Probabilidades:', np.round(prob, 4))
print('Soma:', prob.sum())

Probabilidades: [0.1818 0.1636 0.1455 0.1273 0.1091 0.0909 0.0727 0.0545 0.0364 0.0182]
Soma: 1.0


### 3.3 Roleta e escolha dos pais
A roleta usa a soma acumulada das probabilidades: sorteia-se um número `r` em $[0, 1)$ e escolhe-se o primeiro
indivíduo cuja soma acumulada ultrapassa `r`. São sorteados 5 pares de pais (que geram os 10 filhos), e
os dois pais de um mesmo par são sempre diferentes.

In [11]:
def roleta(prob):
    # Soma acumulada: cada indivíduo ocupa uma faixa de tamanho p_i em [0, 1)
    acumulada = np.cumsum(prob)
    r = rng.random()
    # Primeiro índice cuja soma acumulada é maior que r
    # (o min protege contra arredondamento, quando a soma final fica um pouco abaixo de 1)
    return min(np.searchsorted(acumulada, r, side='right'), len(prob) - 1)

def selecionar_pais(prob, Npares=Nmanter // 2):
    # Sorteia Npares pares de índices de pais (posições na população ordenada)
    pares = []
    for _ in range(Npares):
        pai1 = roleta(prob)
        pai2 = roleta(prob)
        while pai2 == pai1:  # garante dois pais diferentes no mesmo par
            pai2 = roleta(prob)
        pares.append((pai1, pai2))
    return np.array(pares)

### 3.4 Testes da seleção

In [12]:
# Teste 1: a ordenação deixa os custos em ordem crescente e não perde cromossomos
pop_ord, custos_ord = ordenar(pop)
assert np.all(np.diff(custos_ord) >= 0)
assert np.allclose(custos_ord, aptidao(pop_ord))
assert np.array_equal(np.sort(pop_ord, axis=0), np.sort(pop, axis=0))

# Teste 2: probabilidades conforme o enunciado
assert np.isclose(prob.sum(), 1)
assert np.isclose(prob[0], 10/55) and np.isclose(prob[-1], 1/55)

# Teste 3: a frequência da roleta se aproxima das probabilidades
Nsorteios = 100000
freq = np.bincount([roleta(prob) for _ in range(Nsorteios)], minlength=Nmanter) / Nsorteios
assert np.allclose(freq, prob, atol=0.01)

# Teste 4: 5 pares, índices entre 0 e 9, pais diferentes em cada par
for _ in range(1000):
    pares = selecionar_pais(prob)
    assert pares.shape == (5, 2)
    assert pares.min() >= 0 and pares.max() < Nmanter
    assert np.all(pares[:, 0] != pares[:, 1])

print('Todos os testes passaram!')
print('Frequência observada:', np.round(freq, 4))
print('Exemplo de pares (posições no ranking):', selecionar_pais(prob).tolist())

Todos os testes passaram!
Frequência observada: [0.1813 0.1624 0.1443 0.1288 0.1107 0.0912 0.0717 0.0545 0.0369 0.0181]
Exemplo de pares (posições no ranking): [[3, 1], [3, 0], [6, 3], [0, 2], [7, 2]]
